# Seoul Bike Sharing Demand Prediction

**Goal:** Classify whether an hour of the day will have **High Demand** or **Low Demand** for rental bikes in Seoul, using weather and calendar information.

**Approach**
1. Load and inspect the Seoul Bike dataset
2. Clean the data (duplicates, empty columns, missing values)
3. Explore the data visually (EDA)
4. Engineer features (date parts, one-hot encoding)
5. Split into train/test sets and binarize the target using the training median
6. Scale the features
7. Train a baseline model and a Logistic Regression model
8. Evaluate both with accuracy, confusion matrices and a classification report

**Dataset file:** `SeoulBikeData.csv`

## 0. Import Libraries
Libraries for data handling (`pandas`), plotting (`matplotlib`, `seaborn`) and machine learning (`scikit-learn`).

In [1]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

## 1. Loading the Dataset
The CSV is encoded in `cp949` (Korean), so the encoding is specified when reading it. We then print the size, first records, column names, data types and missing values.

In [2]:
data = pd.read_csv("SeoulBikeData.csv", encoding="cp949")

print("Seoul Bike dataset successfully loaded!")
print(f"\nDATASET SIZE: {data.shape[0]} Rows, {data.shape[1]} Columns")

FileNotFoundError: [Errno 2] No such file or directory: 'SeoulBikeData.csv'

In [ ]:
print("FIRST 10 RECORDS:")
data.head(10)

In [ ]:
print("COLUMN NAMES:")
print(data.columns)

print("\nDATASET INFORMATION:")
data.info()

print("\nMISSING VALUES:")
print(data.isnull().sum())

## 2. Cleaning the Dataset & Logging
The data is cleaned in three steps, with a log printed before and after:
1. Drop duplicate rows
2. Drop completely empty columns
3. Drop rows with missing values

In [ ]:
# --- BEFORE CLEANING LOG ---
print("[BEFORE CLEANING]")
print(f"Total Rows: {len(data)}")
print(f"Total Columns: {data.shape[1]}")
print(f"Duplicate Rows Count: {data.duplicated().sum()}")
print("Missing Values Per Column:")
print(data.isnull().sum()[data.isnull().sum() > 0] if data.isnull().sum().sum() > 0 else "  No missing values found.")

In [ ]:
# --- PERFORMING CLEANING ---
# 1. Drop duplicate rows if present
initial_rows = len(data)
data = data.drop_duplicates()
duplicates_removed = initial_rows - len(data)

# 2. Drop completely empty columns
empty_columns = [c for c in data.columns if data[c].isnull().all()]
if empty_columns:
    data = data.drop(columns=empty_columns)

# 3. Drop rows with missing values (if any exist)
missing_before = data.isnull().sum().sum()
data = data.dropna()

# --- AFTER CLEANING LOG ---
print("[AFTER CLEANING]")
print(f"Total Rows: {len(data)} (Removed {duplicates_removed} duplicates)")
print(f"Total Columns: {data.shape[1]} (Dropped empty columns: {empty_columns if empty_columns else 'None'})")
print(f"Duplicate Rows Count: {data.duplicated().sum()}")
print(f"Missing Values Count: {data.isnull().sum().sum()}")
print(f"Final Dataset Shape: {data.shape}")
print("="*40)

## 3. Exploratory Data Analysis (EDA)
Descriptive statistics and visualizations to understand the patterns in bike rentals.

In [ ]:
print("DESCRIPTIVE STATISTICS:")
pd.set_option('display.max_columns', None)
data.describe()

### 3.1 Distribution of Bike Rental Counts
Shows how many bikes are typically rented per hour. The distribution is right-skewed, with many low-demand hours and a few very high ones.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(data["Rented Bike Count"], bins=30, edgecolor="black")
plt.title("Distribution of Bike Rental Counts")
plt.xlabel("Rented Bike Count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 3.2 Average Rentals by Hour
Reveals daily usage patterns, such as commuting peaks in the morning and evening.

In [ ]:
hourly_rentals = data.groupby("Hour")["Rented Bike Count"].mean()
plt.figure(figsize=(10, 5))
plt.bar(hourly_rentals.index, hourly_rentals.values)
plt.title("Average Bike Rentals by Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Average Rented Bikes")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

### 3.3 Average Rentals by Season
Compares rental demand across seasons, from highest to lowest.

In [ ]:
seasonal_rentals = data.groupby("Seasons")["Rented Bike Count"].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
seasonal_rentals.plot(kind="bar")
plt.title("Average Bike Rentals by Season")
plt.xlabel("Season")
plt.ylabel("Average Rented Bikes")
plt.tight_layout()
plt.show()

### 3.4 Relationship between Temperature and Bike Rental Count
A scatter plot showing how rentals change with temperature. Note that the column name contains a garbled degree symbol (`캜`) caused by the `cp949` encoding, so it is referenced exactly as it appears.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    data["Temperature(캜)"],
    data["Rented Bike Count"],
    alpha=0.3
)

plt.title("Relationship between Temperature and Bike Rental Count")
plt.xlabel("Temperature")
plt.ylabel("Rented Bike Count")

plt.tight_layout()
plt.show()

### 3.5 Correlation Matrix Heatmap
Shows linear correlations between all numerical features, helping identify which variables relate most strongly to rental counts.

In [ ]:
plt.figure(figsize=(10, 8))
numeric_data = data.select_dtypes(include=['float64', 'int64'])
correlation = numeric_data.corr()

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    square=True,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()

## 4. Feature Preparation & Engineering
- Keep only records where the bike-sharing system was functioning, then drop the `Functioning Day` column (every remaining value is "Yes").
- Convert `Date` to a datetime and extract **Month** and **Day of Week**.
- Treat `Hour`, `Month` and `Day of Week` as categorical (strings) so they get one-hot encoded.
- Drop `Rented Bike Count` (the target), `Date`, and `Dew point temperature` (highly correlated with temperature) from the features.

In [ ]:
# Keep records where the bike-sharing system is functioning
processed_data = data[data["Functioning Day"] == "Yes"].copy()

# Remove Functioning Day because every remaining value is Yes
processed_data = processed_data.drop(columns=["Functioning Day"])

print("Records after filtering:", len(processed_data))

# Datetime conversion
processed_data["Date"] = pd.to_datetime(
    processed_data["Date"],
    dayfirst=True,
    errors="coerce"
)

processed_data["Month"] = processed_data["Date"].dt.month.astype(str)
processed_data["Day of Week"] = processed_data["Date"].dt.day_of_week.astype(str)
processed_data["Hour"] = processed_data["Hour"].astype(str)

# Select features
X = processed_data.drop(columns=["Rented Bike Count", "Date", "Dew point temperature(캜)"])
y_continuous = processed_data["Rented Bike Count"]

raw_features = list(X.columns)

print(f"Total Base Features Selected ({len(raw_features)}):")
for feat in raw_features:
    print(f"  - {feat}")

## 5. Data Splitting & Target Binarization
- Split the data 80% train / 20% test (`random_state=42` for reproducibility).
- One-hot encode categorical features, then align the test columns to the training columns.
- Compute the **median of the training target only** (to avoid data leakage) and use it as the threshold: hours at or above it are **High Demand (1)**, otherwise **Low Demand (0)**.

In [ ]:
X_train, X_test, y_train_continuous, y_test_continuous = train_test_split(
    X, y_continuous,
    test_size=0.2,
    random_state=42,
)

# One-hot encoding based on training data
X_train = pd.get_dummies(X_train, dtype=int)
X_test = pd.get_dummies(X_test, dtype=int)

# Match test columns to training columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Calculate the target median using ONLY the isolated training split data
train_median = y_train_continuous.median()
print(f"Non-Stratified Training Median Threshold: {train_median}")

# Binarize both targets using the training set's threshold
y_train = (y_train_continuous >= train_median).astype(int)
y_test = (y_test_continuous >= train_median).astype(int)

## 6. Feature Scaling
Standardize features (zero mean, unit variance). The scaler is **fit on the training data only** and then applied to the test data.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. Baseline Model
A `DummyClassifier` that always predicts the most frequent class. Any useful model should beat this accuracy.

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train_scaled, y_train)
dummy_pred = dummy_model.predict(X_test_scaled)
dummy_acc = accuracy_score(y_test, dummy_pred)

print("--- BASELINE MODEL EVALUATION ---")
print(f"Baseline Accuracy: {dummy_acc * 100:.2f}%")

In [ ]:
# Plot Baseline Confusion Matrix
dummy_cm = confusion_matrix(y_test, dummy_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(dummy_cm, display_labels=["Low Demand", "High Demand"]).plot(
    cmap="Blues", ax=ax, colorbar=False
)
plt.title("Baseline Model Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Model Training: Logistic Regression
Train a Logistic Regression classifier on the scaled training data.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

## 9. Model Evaluation
Evaluate Logistic Regression on the test set: accuracy, improvement over the baseline, a confusion matrix, and a per-class classification report (precision, recall, F1).

In [ ]:
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print("--- LOGISTIC REGRESSION EVALUATION ---")
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print(f"Improvement over Baseline: {(accuracy - dummy_acc) * 100:.2f}%")

In [ ]:
# Plot Logistic Regression Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Low Demand", "High Demand"],
    cmap="Blues"
)
plt.title("Confusion Matrix - Bike Rental Demand")
plt.tight_layout()
plt.show()

In [ ]:
print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=["Low Demand", "High Demand"]))